<a href="https://colab.research.google.com/github/raisharad/GenerativeAIandAgenticAI/blob/main/Sharad_UpGrad_HandsOn_0407.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

by Suman Kumar Bera
email id: skbera.iitkgp21@gmail.com<br>
LinkdIn: https://www.linkedin.com/in/skbera4/

### 🤖 Hands-On Tutorial: LLM Prompting and Structured Outputs

**What you'll learn today:**

| Part | Topic |
|---|---|
| 1 | Inference approaches (HuggingFace, HuggingFace API, OpenAI Client) |
| 2 | Prompting (summarization, QA, classification) |
| 3 | Constrained generation & structured outputs (Pydantic, LlamaIndex, and Instructor) |

If you are using

In [ ]:
!pip install -q transformers accelerate huggingface_hub requests regex pydantic instructor openai sentencepiece

In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())
print("Setup complete ✅")

GPU available: False
Setup complete ✅


---
# PART 1 — HuggingFace Inference Approaches + Prompting

## 1.1 Three ways to run an open-source LLM

| Approach | Needs internet? | Needs GPU locally? | Best for |
|---|---|---|---|
| **A. Transformers (local)** | ❌ No | Optional (faster with one) | Full control, fine-tuning |
| **B. HF Inference API (hosted)** | ✅ Yes | ❌ No | Quick prototyping, no setup |
| **C. vLLM (zero-config)** | ✅ Yes | ❌ No | Easiest big-model runs |

**Approach A (Transformers locally on the free GPU)**.

In [ ]:
# Load a small instruction-tuned model (Approach A: Transformers locally)
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"  # open, no gated-access login required
# Alternative (needs HF login + license accept): "google/gemma-2-2b-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="auto"
)

print(f"Loaded {MODEL_ID} on", model.device)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loaded Qwen/Qwen2.5-1.5B-Instruct on cpu


In [ ]:
import os

print(os.path.expanduser("~/.cache/huggingface"))

/root/.cache/huggingface


In [ ]:
# A small helper so we don't repeat boilerplate for the rest of the notebook

def ask(prompt, max_new_tokens=150, system=None, temperature=None):
    """Send a chat-style prompt to our local model and return the text reply."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    gen_kwargs = dict(max_new_tokens=max_new_tokens, do_sample=temperature is not None)
    if temperature:
        gen_kwargs["temperature"] = temperature

    outputs = model.generate(**inputs, **gen_kwargs)
    reply = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return reply.strip()

print("Helper function `ask()` ready!")

Helper function `ask()` ready!


In [ ]:
system = "Explain Simply"
prompt = "What is quantum computing in one sentence?"
messages = []
if system:
    messages.append({"role": "system", "content": system})
messages.append({"role": "user", "content": prompt})
print(messages)

[{'role': 'system', 'content': 'Explain Simply'}, {'role': 'user', 'content': 'What is quantum computing in one sentence?'}]


Q. What is the need for system and user roles?

In [ ]:
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
text

'<|im_start|>system\nExplain Simply<|im_end|>\n<|im_start|>user\nWhat is quantum computing in one sentence?<|im_end|>\n<|im_start|>assistant\n'

Q. What is the need for this type of chat template?

In [ ]:
inputs = tokenizer(text, return_tensors="pt").to(model.device)
inputs

{'input_ids': tensor([[151644,   8948,    198,    840,  20772,  28424, 151645,    198, 151644,
            872,    198,   3838,    374,  30128,  24231,    304,    825,  11652,
             30, 151645,    198, 151644,  77091,    198]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [ ]:
temperature = 0.3
gen_kwargs = dict(max_new_tokens=80, do_sample=temperature is not None)
gen_kwargs


{'max_new_tokens': 80, 'do_sample': True}

In [ ]:
outputs = model.generate(**inputs, **gen_kwargs)

In [ ]:
reply = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
reply

'Quantum computing uses the principles of quantum mechanics to perform complex calculations much faster than traditional computers.'

In [ ]:
# using local model downoaded from HuggingFace , Qwen

article = """
Antibiotics are a type of medication used to treat bacterial infections. They work by either
killing the bacteria or preventing them from reproducing, allowing the body's immune system to
fight off the infection. Antibiotics are usually taken orally in the form of pills, capsules, or
liquid solutions, or sometimes administered intravenously. They are not effective against viral
infections, and using them inappropriately can lead to antibiotic resistance.
"""

prompt = f"Summarize the following text in exactly one sentence:\n\n{article}"
summary = ask(prompt, max_new_tokens=80, temperature=0.3)
print("SUMMARY:\n", summary)

SUMMARY:
 Antibiotics are medications that kill bacteria or prevent their reproduction, treating bacterial infections but not viruses, emphasizing proper use to avoid developing antibiotic-resistant strains.


## 1.2 Approach B: HuggingFace hosted Inference API

No GPU or model download needed — just an HTTP request. Useful when you don't want to load a model into memory at all. (Requires a free token from huggingface.co/settings/tokens — **skip this cell if you don't have one**, it's optional context.)

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("hf_key")

In [ ]:
import os
from openai import OpenAI

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=HF_TOKEN, #you can directly paste your token here
)

completion = client.chat.completions.create(
    model="Qwen/Qwen2.5-1.5B-Instruct:featherless-ai",
    messages=[
        {
            "role": "user",
            "content": "What is the capital of France?"
        }
    ],
)

print(completion.choices[0].message)

ChatCompletionMessage(content='The capital of France is Paris.\n', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)


## Approach 3: UpGrad hosted EndPoint

In [ ]:
from google.colab import userdata

API_KEY = userdata.get("api_key")

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://103.42.50.41:443/api1/",
    api_key=API_KEY, #you can directly paste your token here
)

DEFAULT_MODEL = "Qwen/Qwen2.5-7B-Instruct-AWQ"

try:
    response = client.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=[
            {"role": "user", "content": "Reply with exactly: MODEL_WORKING"}
        ],
        max_tokens=10,
        temperature=0
    )

    print("✅ Model responded")
    print(response.choices[0].message.content)

except Exception as e:
    print("❌ Error")
    print(type(e).__name__)
    print(e)

✅ Model responded
MODEL_WORKING


In [ ]:
def ask_Qwen_7B(prompt, max_tokens=150, system=None, temperature=0):
    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    response = client.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=messages,
        max_tokens=max_tokens,
        temperature=temperature,
    )

    return response.choices[0].message.content.strip()

In [ ]:
print(ask_Qwen_7B("What is the capital of France?"))

The capital of France is Paris.


| Local transformers        | OpenAI API                          |
|----------------------------|--------------------------------------|
| `apply_chat_template()`    | server does it                      |
| `tokenizer()`               | server does it                      |
| `generate()`                | server does it                      |
| `decode()`                  | server does it                      |
| helper function optional   | helper function optional in chat    |

## 1.1 Prompting Task 1 — Classification

- **Zero-shot sentiment classification**


In [ ]:
classification_prompt = "Classify sentiment as exactly one word (positive/negative/neutral): 'The food was cold but the service was excellent.'"

print("temp=0.0:", [ask_Qwen_7B(classification_prompt, temperature=0.0, max_tokens=5) for _ in range(5)])
# print("temp=1.0:", [ask_Qwen_7B(classification_prompt, temperature=1.0, max_tokens=5) for _ in range(5)])
# print("temp=2.0:", [ask_Qwen_7B(classification_prompt, temperature=2.0, max_tokens=5) for _ in range(5)])
print("temp=3.0:", [ask_Qwen_7B(classification_prompt, temperature=3.0, max_tokens=5) for _ in range(5)])
# print("temp=4.0:", [ask_Qwen_7B(classification_prompt, temperature=4.0, max_tokens=5) for _ in range(5)])
print("temp=5.0:", [ask_Qwen_7B(classification_prompt, temperature=5.0, max_tokens=5) for _ in range(5)])


temp=0.0: ['Neutral', 'Neutral', 'Neutral', 'Neutral', 'Neutral']
temp=3.0: ['mixed', 'Mixed', 'Mixed', 'Neutral', '-neutral']
temp=5.0: ['Neutral', '-neutral', 'Positive', 'Neutral', '混合']


In [ ]:
prompt = "Write one creative sentence describing a rainy city street."

temperatures = [0.0, 0.3, 0.7, 1.2, 1.9]

for temp in temperatures:
    print(f"\n=== temperature = {temp} ===")
    for run in range(3):  # run 3 times at each temp to show variance (or lack of it)
        print(f"  [{run+1}] {ask_Qwen_7B(prompt, temperature=temp)}")


=== temperature = 0.0 ===
  [1] The rain danced elegantly down the cobblestone streets, painting each puddle with shimmering reflections of neon signs and whispering secrets to the passersby in a language only the city and the downpour could understand.
  [2] The rain danced elegantly down the cobblestone streets, painting each puddle with shimmering reflections of neon signs and whispering secrets to the passersby in a language only the city and the downpour could understand.
  [3] The rain danced elegantly down the cobblestone streets, painting each puddle with shimmering reflections of neon signs and whispering secrets to the passersby in a language only the city and the downpour could understand.

=== temperature = 0.3 ===
  [1] The rain danced elegantly down the cobblestone streets, painting each puddle with shimmering silver brushstrokes as if the city itself was a vast canvas come to life.
  [2] The rain danced elegantly down the cobblestone streets, painting each puddle with s

## 1.2 Prompting Task 2 — Text Summarization

Two common summarization styles:
- **Zero-shot instruction summarization** (give context + clear instruction)
- **TL;DR style** (ultra-short, audience-aware)

In [ ]:
summary_prompt = """
Summarize the following article in 2-3 sentences.

Article:
Large language models (LLMs) are deep learning models trained on massive amounts
of text data. They can perform tasks such as text generation, summarization,
translation, question answering, and code generation. Recent advances in
transformer architectures and scaling laws have significantly improved their
performance. However, LLMs can still generate incorrect information, known as
hallucinations, and require careful prompting and evaluation before being
deployed in critical applications.
"""

print(ask_Qwen_7B(summary_prompt, max_tokens=100, temperature=0.0))

Large language models (LLMs) are advanced deep learning models trained on extensive text data, capable of various tasks including text generation and translation, but they can produce incorrect information (hallucinations) and thus need careful prompting and evaluation for critical applications.


In [ ]:
print("temperature=0.0")
print(ask_Qwen_7B(summary_prompt, max_tokens=100, temperature=0.0))

print("\ntemperature=0.8")
print(ask_Qwen_7B(summary_prompt, max_tokens=100, temperature=2.0))

temperature=0.0
Large language models (LLMs) are advanced deep learning models trained on extensive text data, capable of various tasks including text generation and translation, but they can produce incorrect information (hallucinations) and thus need careful prompting and evaluation for critical applications.

temperature=0.8
Large language models (LLMs) are advanced deep learning systems trained on extensive text data, capable of performing various tasks including text generation and translation; while their performance has significantly improved due to recent technological advancements, they can still produce inaccurate or hallucinatory outputs, emphasizing the need for careful prompting and thorough evaluation, especially in critical applications.


In [ ]:
LEGAL_TEXT = """
A tenant must pay rent on or before the fifth day of each month.
If the rent is paid more than ten days after the due date without prior written approval from the landlord,
a late fee of $50 shall be charged.
"""

PROMPT = f"""
Read the following legal clause.

{LEGAL_TEXT}

Tasks:
1. Summarize the clause in at most two bullet points.
2. Generate a Python function

def calculate_late_fee(days_late):

that returns the appropriate late fee according to the clause.
"""

print(ask_Qwen_7B(PROMPT, max_tokens=200, temperature=0.0))

### Summary of the Clause
- Rent must be paid by the 5th of each month.
- A late fee of $50 is charged if rent is paid more than 10 days after the due date without prior written approval.

### Python Function to Calculate Late Fee

```python
def calculate_late_fee(days_late):
    """
    Calculates the late fee based on the number of days late.
    
    Parameters:
        days_late (int): The number of days the rent is paid after the due date.
        
    Returns:
        int: The late fee amount.
    """
    if days_late > 10:
        return 50
    else:
        return 0
```

This function checks if the rent is paid more than 10 days late and returns the late fee accordingly.


## 1.3 Prompting Task 3 — Question Answering

Two flavors:
- **Context-bound QA**: answer must come only from the given context (good for RAG / policy lookups)
- **Open-domain synthesis QA**: model uses its own knowledge to answer + format a list

In [ ]:
context = """
Our company policy states that employees get $500/year for professional development.
Reimbursement requests must be filed at least 30 days prior to the course start date.
"""
question = "Can I get reimbursed for a $200 course I bought yesterday without prior approval?"

prompt = f"""Answer the question using ONLY the context below. Keep it short. If unsure, say "Unsure about answer".

Context: {context}

Question: {question}

Answer:"""

print(ask_Qwen_7B(prompt, max_tokens=150, temperature=2.0))
print(ask(prompt, max_new_tokens=150))

Unsure about answer.
No, you cannot get reimbursed for a $200 course you bought yesterday without prior approval. The reimbursement request must be filed at least 30 days prior to the course start date according to your company's policy.


In [ ]:
# Open-domain, formatted output
prompt = "List the three primary constraints of the James Webb Space Telescope's cooling system. Format as a numbered list."
print(ask(prompt, max_new_tokens=150))
print(ask_Qwen_7B(prompt, max_tokens=150))

1. **Thermal Insulation**: The telescope must be extremely well-insulated to prevent heat from entering its sensitive instruments and cameras.
2. **Radiation Shielding**: It needs an extensive shield against solar radiation to maintain optimal operating temperatures for scientific observations.
3. **Active Cooling System**: A sophisticated active cooling system is employed to actively manage the temperature of critical components within the telescope.
1. The need to maintain extremely low temperatures to reduce thermal noise and improve the telescope's sensitivity.
2. Limited supply of cryogenic fluids, which are essential for maintaining these low temperatures.
3. Design limitations that restrict the size and complexity of heat sinks and radiators used for cooling.


### Reasoning

In [ ]:
# --- A reasoning problem that trips up zero-shot ---
problem = """A juggler has 16 balls. Half of the balls are golf balls,
and half of the golf balls are blue. How many blue golf balls are there? Give the answer directly."""

# ---------------------------------------------------------
# 1. ZERO-SHOT: just ask directly
# ---------------------------------------------------------
zero_shot_prompt = f"Question: {problem}\nAnswer:"

zero_shot_answer = ask(zero_shot_prompt, max_new_tokens=150, temperature=0.0)
print("Zero-shot answer:", zero_shot_answer)
# Smaller models often blurt "8" (they compute 16/2 and stop),
# instead of the correct 16/2/2 = 4.

Zero-shot answer: 8


In [ ]:
# --- A reasoning problem that trips up zero-shot ---
problem = """A juggler has 16 balls. Half of the balls are golf balls,
and half of the golf balls are blue. How many blue golf balls are there? Think step by step and then give the answer."""

# ---------------------------------------------------------
# 1. ZERO-SHOT: just ask directly
# ---------------------------------------------------------
zero_shot_prompt = f"Question: {problem}\nAnswer:"

zero_shot_answer = ask(zero_shot_prompt, max_new_tokens=150)
print("Zero-shot answer:", zero_shot_answer)
# Smaller models often blurt "8" (they compute 16/2 and stop),
# instead of the correct 16/2/2 = 4.

Zero-shot answer: Let's break this down step by step:

1. The juggler has 16 balls in total.

2. Half of these balls are golf balls:
   \[
   \frac{1}{2} \times 16 = 8 \text{ golf balls}
   \]

3. Out of these 8 golf balls, half of them are blue:
   \[
   \frac{1}{2} \times 8 = 4 \text{ blue golf balls}
   \]

Therefore, there are 4 blue golf balls.


In [ ]:
zero_shot_prompt = """
Determine whether the sequence should be labeled ALPHA or BETA.

Sequence:
2, 5, 8, 11

Answer:
"""

print(ask_Qwen_7B(zero_shot_prompt, temperature=0.0, max_tokens=220))

To determine whether the sequence should be labeled ALPHA or BETA, we need to identify the pattern in the sequence and see if it fits either label.

The given sequence is: 2, 5, 8, 11

Let's analyze the pattern:
- The difference between the first term (2) and the second term (5) is 3.
- The difference between the second term (5) and the third term (8) is 3.
- The difference between the third term (8) and the fourth term (11) is 3.

This indicates that the sequence is increasing by 3 each time. 

Without specific definitions for what ALPHA and BETA sequences are, we can infer that a common type of sequence like this might be labeled based on its arithmetic progression. However, since the problem does not provide explicit rules for labeling ALPHA or BETA, we can only describe the sequence as an arithmetic sequence with a common difference of 3.

If we assume that any sequence with a constant difference is labeled BETA (a common pattern in such problems


In [ ]:
few_shot_prompt = """
Determine whether the sequence should be labeled ALPHA or BETA.

Example 1
Sequence:
1, 4, 7, 10
Answer: ALPHA

Example 2
Sequence:
3, 6, 9, 12
Answer: ALPHA

Example 3
Sequence:
2, 4, 8, 16
Answer: BETA

Example 4
Sequence:
5, 10, 20, 40
Answer: BETA

Now classify:

Sequence:
2, 5, 8, 11

Answer:
"""

print(ask_Qwen_7B(few_shot_prompt, temperature=0.0, max_tokens=120))

Answer: ALPHA

The sequence 2, 5, 8, 11 follows an arithmetic pattern where each term increases by a constant difference of 3. This is characteristic of an ALPHA sequence.


| Task                         | Recommended Temperature |
|-------------------------------|--------------------------|
| Classification                | 0                        |
| Extraction                    | 0                        |
| Summarization                 | 0–0.3                    |
| Coding                        | 0–0.2                    |
| Reasoning (self-consistency)  | 0.5–0.8                  |
| Brainstorming                 | 0.8–1.2                  |
| Creative writing              | 1.0+                     |

---
# PART 2 — Constrained Generation & Structured Outputs

LLMs generate text **token by token**, picking from a probability distribution (logits → softmax) over the whole vocabulary at every step. **Constrained generation** restricts which tokens are even allowed at each step, so the output is *guaranteed* to follow a structure — instead of hoping the model "behaves."

We'll go from the lowest level (manually masking logits) up to the easiest practical tool (Pydantic + Instructor).

### PyDantic

In [ ]:
!pip install llama-index
!pip install llama-index-readers-file
!pip install -q llama-index-llms-openai-like
!pip install -q instructor

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.5/164.5 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 12.8 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: nltk
    Found existing installation: nltk 3.9.1
    Uninstalling nltk-3.9.1:
      Successfully uninstalled nltk-3.9.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, w

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353.9/353.9 kB 24.1 MB/s eta 0:00:00


In [ ]:
from llama_index.readers.file import PDFReader
from pathlib import Path

In [ ]:
pdf_reader = PDFReader()
documents = pdf_reader.load_data(file=Path("/content/wordpress-pdf-invoice-plugin-sample.pdf"))
text = documents[0].text
print(text)

Invoice
Payment is due within 30 days from date of invoice. Late payment is subject to fees of 5% per month.
Thanks for choosing DEMO - Sliced Invoices | admin@slicedinvoices.com
Page 1/1
From:
DEMO - Sliced Invoices
Suite 5A-1204
123 Somewhere Street
Your City AZ 12345
admin@slicedinvoices.com
Invoice Number INV-3337
Order Number 12345
Invoice Date January 25, 2016
Due Date January 31, 2016
Total Due $93.50
To:
Test Business
123 Somewhere St
Melbourne, VIC 3000
test@test.com
Hrs/Qty Service Rate/Price Adjust Sub Total
1.00 Web Design
This is a sample description... $85.00 0.00% $85.00
Sub Total $85.00
Tax $8.50
Total $93.50
ANZ Bank
ACC # 1234 1234
BSB # 4321 432 Paid


In [ ]:
from google.colab import userdata

API_KEY = userdata.get("api_key")

## A. Import Libraries

In [ ]:
from pathlib import Path
from datetime import datetime
from typing import List
from pydantic import BaseModel, Field, field_validator
from dateutil import parser as date_parser

### B. Load the PDF

In [ ]:
# ── Load the PDF text ───────────────────────────────────────────────────────
pdf_reader = PDFReader()
documents = pdf_reader.load_data(file=Path("/content/wordpress-pdf-invoice-plugin-sample.pdf"))
text = documents[0].text
print(text)

Invoice
Payment is due within 30 days from date of invoice. Late payment is subject to fees of 5% per month.
Thanks for choosing DEMO - Sliced Invoices | admin@slicedinvoices.com
Page 1/1
From:
DEMO - Sliced Invoices
Suite 5A-1204
123 Somewhere Street
Your City AZ 12345
admin@slicedinvoices.com
Invoice Number INV-3337
Order Number 12345
Invoice Date January 25, 2016
Due Date January 31, 2016
Total Due $93.50
To:
Test Business
123 Somewhere St
Melbourne, VIC 3000
test@test.com
Hrs/Qty Service Rate/Price Adjust Sub Total
1.00 Web Design
This is a sample description... $85.00 0.00% $85.00
Sub Total $85.00
Tax $8.50
Total $93.50
ANZ Bank
ACC # 1234 1234
BSB # 4321 432 Paid


In [ ]:
content = text

prompt = f"this is the bill: {content} tell me if the bill has been paid or not?"
print(ask_Qwen_7B(prompt, temperature=0.0))

Based on the information provided in the invoice, there is no indication that the bill has been paid. The invoice clearly states that payment is due within 30 days from the invoice date, which was January 25, 2016. The due date was January 31, 2016. Since no payment details or confirmation of payment are included in your message, it appears that the bill has not yet been paid.

If you need to confirm whether the payment has been made, you would typically look for a payment receipt or confirmation from the payer. If you have not received any such confirmation, it is safe to assume that the payment has not been made as of yet.

Would you like assistance in sending a follow


### C. PyDantic Class Creation

In [ ]:
# ── Schema ───────────────────────────────────────────────────────────────
# Using real `datetime` fields is fine for both approaches below, since neither
# relies on the server's native tool-calling JSON-schema parser (that's what
# broke earlier with $defs/$ref). But the model can still return dates in
# inconsistent formats ("January 25, 2016" vs "2016-01-25"), so we add a
# validator to normalize whatever string comes back before Pydantic parses it
# into a datetime — this is what prevents the ValueError you hit before.
class Invoice(BaseModel):
    """A representation of information from an invoice."""

    vendor: str
    invoice_date: datetime
    due_date: datetime
    invoice_number: str
    items: List[str]

    # before validating the field, call this function
    @field_validator("invoice_date", "due_date", mode="before") #tells Pydantic when to run your validation function.
    @classmethod #tells Python how to call that function
    def normalize_date(cls, v):
        if isinstance(v, str):
            return date_parser.parse(v)
        return v

### D. LlamaIndex

In [ ]:
# ── Option A: llama-index, using text-based structured prediction ──────────
from llama_index.llms.openai_like import OpenAILike
from llama_index.core.program import LLMTextCompletionProgram
from llama_index.core.output_parsers import PydanticOutputParser

llm = OpenAILike(
    model="Qwen/Qwen2.5-7B-Instruct-AWQ",
    api_base="http://103.42.50.41:443/api1/",
    api_key=API_KEY,
    is_chat_model=True,
)

output_parser = PydanticOutputParser(output_cls=Invoice)

program = LLMTextCompletionProgram.from_defaults(
    output_parser=output_parser,
    prompt_template_str=(
        "Extract the invoice information from the text below.\n"
        "{format_instructions}\n\n"
        "Text:\n{text}\n"
    ),
    llm=llm,
    verbose=True,
)

response = program(
    text=text,
    format_instructions=output_parser.get_format_string(),
)
print("llama-index result:")
print(response)

llama-index result:
vendor='DEMO - Sliced Invoices' invoice_date=datetime.datetime(2016, 1, 25, 0, 0, tzinfo=tzlocal()) due_date=datetime.datetime(2016, 1, 31, 0, 0, tzinfo=tzlocal()) invoice_number='INV-3337' items=['Web Design This is a sample description... $85.00']


In [ ]:
print(response.model_dump_json(indent=2))

{
  "vendor": "DEMO - Sliced Invoices",
  "invoice_date": "2016-01-25T00:00:00Z",
  "due_date": "2016-01-31T00:00:00Z",
  "invoice_number": "INV-3337",
  "items": [
    "Web Design This is a sample description... $85.00"
  ]
}


### E. Instructor

In [ ]:
# ── Option B: instructor, JSON mode ─────────────────────────────────────
import instructor
from openai import OpenAI

client = OpenAI(
    base_url="http://103.42.50.41:443/api1/",
    api_key=API_KEY,
)

instructor_client = instructor.from_openai(client, mode=instructor.Mode.JSON)

extracted_invoice: Invoice = instructor_client.chat.completions.create(
    model="Qwen/Qwen2.5-7B-Instruct-AWQ",
    response_model=Invoice,
    messages=[
        {"role": "user", "content": f"Extract the invoice details from this text:\n\n{text}"}
    ],
)

print("\ninstructor result:")
print(extracted_invoice)

# No manual parsing needed anymore -- these are already real datetime objects,
# normalized by the field_validator regardless of what string format the
# model originally returned.
print("\nInvoice date:", extracted_invoice.invoice_date)
print("Due date:", extracted_invoice.due_date)
print(type(extracted_invoice.invoice_date))


instructor result:
vendor='DEMO - Sliced Invoices' invoice_date=datetime.datetime(2016, 1, 25, 0, 0, tzinfo=tzlocal()) due_date=datetime.datetime(2016, 1, 31, 0, 0, tzinfo=tzlocal()) invoice_number='INV-3337' line_items=[LineItem(item_name='Web Design', quantity=1.0, price=85.0, service_type=<ServiceType.web_design: 'web_design'>)] total_amount=93.5 is_paid=True

Invoice date: 2016-01-25 00:00:00+00:00
Due date: 2016-01-31 00:00:00+00:00
<class 'datetime.datetime'>


In [ ]:
extracted_invoice.invoice_number

'INV-3337'

### F. USE CASE(to get the pending or paid amount and type of service)

In [ ]:
from typing import Optional
from enum import Enum

class ServiceType(str, Enum):
    """Constrain to a fixed set of categories so the model can't invent new ones."""
    web_design = "web_design"
    consulting = "consulting"
    development = "development"
    hosting = "hosting"
    marketing = "marketing"
    other = "other"

class LineItem(BaseModel):
    item_name: str = Field(description="Just the name of the service/product, no price or description")
    quantity: float = Field(description="Quantity or hours")
    price: float = Field(description="Price for this line item")
    service_type: ServiceType = Field(description="The category of service this line item represents")

class Invoice(BaseModel):
    vendor: str
    invoice_date: datetime
    due_date: datetime
    invoice_number: str
    line_items: List[LineItem]
    total_amount: float = Field(description="The final total amount due on the invoice, including tax")
    is_paid: bool = Field(description="Whether the invoice has already been paid, based on any payment status indicators in the text")

    # before validating the field, call this function
    @field_validator("invoice_date", "due_date", mode="before")
    @classmethod
    def normalize_date(cls, v):
        if isinstance(v, str):
            return date_parser.parse(v)
        return v

In [ ]:
extracted_invoice: Invoice = instructor_client.chat.completions.create(
    model="Qwen/Qwen2.5-7B-Instruct-AWQ",
    response_model=Invoice,
    messages=[
        {"role": "user", "content": f"Extract the invoice details from this text:\n\n{text}"}
    ],
)

print(extracted_invoice)
print("\nTotal amount:", extracted_invoice.total_amount)
print("Is paid:", extracted_invoice.is_paid)
for item in extracted_invoice.line_items:
    print(f"- {item.item_name} ({item.service_type.value}): {item.quantity} x ${item.price}")

vendor='DEMO - Sliced Invoices' invoice_date=datetime.datetime(2016, 1, 25, 0, 0, tzinfo=tzlocal()) due_date=datetime.datetime(2016, 1, 31, 0, 0, tzinfo=tzlocal()) invoice_number='INV-3337' line_items=[LineItem(item_name='Web Design', quantity=1.0, price=85.0, service_type=<ServiceType.web_design: 'web_design'>)] total_amount=93.5 is_paid=True

Total amount: 93.5
Is paid: True
- Web Design (web_design): 1.0 x $85.0


In [ ]:
print(extracted_invoice.model_dump_json(indent=2))

{
  "vendor": "DEMO - Sliced Invoices",
  "invoice_date": "2016-01-25T00:00:00Z",
  "due_date": "2016-01-31T00:00:00Z",
  "invoice_number": "INV-3337",
  "line_items": [
    {
      "item_name": "Web Design",
      "quantity": 1.0,
      "price": 85.0,
      "service_type": "web_design"
    }
  ],
  "total_amount": 93.5,
  "is_paid": true
}


---
# 🎓 Wrap-up

You've now hands-on practiced:

1. **Loading & prompting** open LLMs locally on Colab, for summarization, QA, and classification
2. **Constraining generation** at the practical way (Pydantic + Instructor)
